<a href="https://colab.research.google.com/github/MRigoni10/customer-segmentation-rfm/blob/main/notebooks/customer_segmentation_rfm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Download the official zip file directly into the session
!wget -O online_retail_II.zip "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"

# 2. Extract the compressed file
!unzip -o online_retail_II.zip

# 3. Check the present files
!ls -lh

--2026-09-04 14:58:42--  https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘online_retail_II.zip’

online_retail_II.zi     [       <=>          ]  43.51M  33.5MB/s    in 1.3s    

2026-09-04 14:58:44 (33.5 MB/s) - ‘online_retail_II.zip’ saved [45622418]

Archive:  online_retail_II.zip
 extracting: online_retail_II.xlsx   
total 88M
-rw------- 1 root root  44M May 22  2023 online_retail_II.xlsx
-rw-r--r-- 1 root root  44M Sep  4 14:58 online_retail_II.zip
drwxr-xr-x 1 root root 4.0K Aug 24 13:21 sample_data


In [ ]:
import pandas as pd

print("Loading data...")
# The dataset is divided into two sheets by year: read them and concatenate them together
df_2009 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')

df = pd.concat([df_2009, df_2010], ignore_index=True)
print(f"Dataset loaded successfully! Dimensions: {df.shape}")
df.head()

Loading data...
Dataset loaded successfully! Dimensions: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
# 3. Import libraries
import datetime as dt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# Visual settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
def clean_retail_data(data: pd.DataFrame) -> pd.DataFrame:
  df_clean = data.copy()

  # 1. Handle essential missing values
  # Transactions without a Customer ID do not allow tracking of individual users
  df_clean = df_clean.dropna(subset=["Customer ID"])
  df_clean["Customer ID"] = df_clean["Customer ID"].astype(int).astype(str)

  # 2. Remove cancellations / reversals (Invoice starting with 'C' or negative Quantity)
  df_clean["Invoice"] = df_clean["Invoice"].astype(str)
  df_clean = df_clean[~df_clean["Invoice"].str.startswith("C")]
  df_clean = df_clean[df_clean["Quantity"] > 0]

  # 3. Clean prices and accounting codes (e.g. POST, D, CRATE, M)
  non_product_codes = ["POST", "D", "M", "PADS", "CRATE", "BANK CHARGES"]
  df_clean = df_clean[df_clean["Price"] > 0]
  df_clean = df_clean[~df_clean["StockCode"].isin(non_product_codes)]

  # 4. Reconcile dates and calculate Revenue
  df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
  df_clean["TotalPrice"] = df_clean["Quantity"] * df_clean["Price"]

  return df_clean

df_clean = clean_retail_data(df)
print(f"Original rows: {len(df):,} | Cleaned rows: {len(df_clean):,}")
df_clean.head()

Original rows: 1,067,371 | Cleaned rows: 802,948


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [ ]:
# Snapshot date: 1 day after the last recorded purchase
snapshot_date = df_clean["InvoiceDate"].max() + dt.timedelta(days=1)

rfm = (
    df_clean.groupby("Customer ID")
    .agg({
        "InvoiceDate": lambda x: (snapshot_date - x.max()).days,  # Recency
        "Invoice": "nunique",  # Frequency
        "TotalPrice": "sum",  # Monetary
    })
    .reset_index()
)

rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]

# Security filter on customers with spend > 0
rfm = rfm[rfm["Monetary"] > 0]
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,5633.32
2,12348,75,5,1658.40
3,12349,19,3,3678.69
4,12350,310,1,294.40


In [ ]:
# 1. Skewness correction and standardization
rfm_features = rfm[["Recency", "Frequency", "Monetary"]]
rfm_log = np.log1p(rfm_features)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# 2. Optimization of the number of Clusters k
k_range = range(2, 9)
inertias = []
silhouettes = []

for k in k_range:
  km = KMeans(n_clusters=k, random_state=42, n_init=10)
  labels = km.fit_predict(rfm_scaled)
  inertias.append(km.inertia_)
  silhouettes.append(silhouette_score(rfm_scaled, labels))

# 3. Final model training (usually k=4 for RFM)
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)


print(rfm)

     CustomerID  Recency  Frequency  Monetary  Cluster
0         12346      326         12  77556.46        0
1         12347        2          8   5633.32        0
2         12348       75          5   1658.40        3
3         12349       19          3   3678.69        2
4         12350      310          1    294.40        1
...         ...      ...        ...       ...      ...
5857      18283        4         22   2730.70        0
5858      18284      432          1    461.68        1
5859      18285      661          1    427.00        1
5860      18286      477          2   1296.43        3
5861      18287       43          7   4182.99        0

[5862 rows x 5 columns]


In [ ]:
# Cluster means on real (untransformed) values
cluster_summary = (
    rfm.groupby("Cluster")
    .agg(
        Recency_Mean=("Recency", "mean"),
        Frequency_Mean=("Frequency", "mean"),
        Monetary_Mean=("Monetary", "mean"),
        Customer_Count=("CustomerID", "count"),
    )
    .round(2)
)

# Calculation of percentage share of total revenue
cluster_summary["Revenue_Share_%"] = (
    (
        rfm.groupby("Cluster")["Monetary"].sum()
        / rfm["Monetary"].sum()
        * 100
    ).round(2)
)
display(cluster_summary)

,Recency_Mean,Frequency_Mean,Monetary_Mean,Customer_Count,Revenue_Share_%
Cluster,,,,,
0,28.07,19.18,10876.12,1188,73.99
1,394.78,1.37,319.76,1962,3.59
2,28.30,3.03,854.42,1249,6.11
3,228.94,5.05,1946.89,1463,16.31


In [ ]:
# Interactive 3D Visualization
fig = px.scatter_3d(
    rfm,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Cluster",
    log_x=True,
    log_y=True,
    log_z=True,
    opacity=0.7,
    title="Customer Segmentation: 3D RFM Clusters (Log Scale)",
    hover_data=["CustomerID"],
)
fig.show()